# 20. Why confidence matters for prediction

Generate a sample of 1000 events distributed as a gaussian with mean 125.44 and sigma 0.14.
Extract the 95% CL interval on the mean parameter, extracted with a maximum likelihood fit.

Repeat the procedure in 50 pseudo-experiments and make a plot of the confidence interval extracted in each pseudo experiment as a function of the pseudo-experiment number. The plot should also include a line representing the true mean value. 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Problem setup
true_mu = 125.
true_sigma = 2.
n_events = 500
n_pseudo = 50
cl = 0.95
z = 1.96  # Two-sided 95% quantile for a Gaussian

In [ ]:
rng = np.random.default_rng(12345)

pseudo_ids = np.arange(1, n_pseudo + 1)
mu_hat = np.empty(n_pseudo)
ci_low = np.empty(n_pseudo)
ci_high = np.empty(n_pseudo)

for i in range(n_pseudo):
    sample = rng.normal(loc=true_mu, scale=true_sigma, size=n_events)

    # MLE for Gaussian mean is the sample mean
    mu_mle = sample.mean()
    sigma_mle = sample.std(ddof=0)  # MLE estimate of sigma
    se_mu = sigma_mle / np.sqrt(n_events)

    half_width = z * se_mu
    mu_hat[i] = mu_mle
    ci_low[i] = mu_mle - half_width
    ci_high[i] = mu_mle + half_width

coverage = np.mean((ci_low <= true_mu) & (true_mu <= ci_high))
print(f"Empirical coverage over {n_pseudo} pseudo-experiments: {coverage:.3f}")

In [ ]:
plt.figure(figsize=(8, 6))

# Identify intervals that do not contain the true value
missed = (true_mu < ci_low) | (true_mu > ci_high)
hit = ~missed

# Plot intervals that contain the true value in blue
for i in np.where(hit)[0]:
    plt.errorbar(
        pseudo_ids[i],
        mu_hat[i],
        yerr=[[mu_hat[i] - ci_low[i]], [ci_high[i] - mu_hat[i]]],
        fmt='o',
        ms=4,
        capsize=3,
        color='tab:blue',
        ecolor='tab:blue',
        elinewidth=1,
        alpha=0.9
    )

# Plot intervals that miss the true value in red
for i in np.where(missed)[0]:
    plt.errorbar(
        pseudo_ids[i],
        mu_hat[i],
        yerr=[[mu_hat[i] - ci_low[i]], [ci_high[i] - mu_hat[i]]],
        fmt='o',
        ms=4,
        capsize=3,
        color='tab:red',
        ecolor='tab:red',
        elinewidth=1,
        alpha=0.9
    )

# Add legend entries
plt.errorbar([], [], fmt='o', ms=4, color='tab:blue', ecolor='tab:blue', label='Contains true value', capsize=3)
plt.errorbar([], [], fmt='o', ms=4, color='tab:red', ecolor='tab:red', label='Does not contain true value', capsize=3)

plt.axhline(true_mu, color='tab:green', linestyle='--', linewidth=2, label='True mass')
plt.xlabel('Pseudo-experiment number', loc='right', fontsize=14)
plt.ylabel('Measured Higgs boson mass [GeV]', loc='top', fontsize=14)
plt.title('')
plt.ticklabel_format(axis='y', style='plain', useOffset=False)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Intervals that missed the true value: {np.sum(missed)}/{n_pseudo}")